# R2 · 3 · Redes de grafo: escala por UF (B1), pré-treino nas regiões (B2), quantis altos (D1), cabeça do total (D3)

**Antes de rodar (uma vez só)**, em `MyDrive/dengue_V4/` (que já tem tudo das rodadas anteriores, inclusive `semanal_lib.py`):
1. suba **`r2_lib.py`** (pasta `data_imdc_2026/RODADA2/`);
2. **crie à mão** as pastas `resultados_semanal_r2`, `resultados_uf_r2`, `cache_r2` e `avaliacao_r2` (evita pastas duplicadas quando há duas sessões abertas).

**Nada antigo é alterado.** Os modelos novos gravam em `resultados_semanal_r2/` e `resultados_uf_r2/`; as tabelas novas vão para `avaliacao_r2/` com a data e a hora no nome (`..._AAAAMMDD-HHMM.csv`) e uma cópia `..._ultimo.csv`. As tabelas da rodada anterior (`resultados_semanal_avaliacao/`) continuam como estão.

**Retomada:** se a internet ou a energia cair, reconecte e **rode tudo de novo**. Cada modelo × rodada/temporada vira um arquivo assim que termina (gravado como `.tmp` e renomeado); os treinos longos das redes gravam um checkpoint a cada 5 épocas e continuam da última época salva; o tsfresh grava uma UF por vez; o Optuna grava o estudo no Drive a cada tentativa; o Chronos ajustado fica salvo e não é refeito.

GPU L4 (ou A100). Temporada: ~3–6 h; semanal: ~1–2 h. As redes gravam checkpoint a cada 5 épocas em `cache_r2/ckpt/`; o pré-treino nas 435 regiões é feito uma vez por temporada e semente e reaproveitado.

Todas as versões de temporada partem do melhor isolado da rodada anterior (**MTGNN com peso pela população**, alvo âncora) e mudam uma coisa por vez, mais uma versão com tudo junto.


In [ ]:
NOTEBOOK = 'r2_redes_gpu'
PACOTES = ''

# @title 1 · Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/r2', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'uf_lib.py', 'semanal_lib.py', 'r2_lib.py', 'dados', 'dados_uf',
                 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    if PACOTES:
        r = subprocess.run(f'pip install -q {PACOTES}', shell=True, capture_output=True, text=True)
        print(r.stdout[-400:], r.stderr[-400:])
sys.path.insert(0, PASTA_LOCAL)
PASTA_SEM_R2 = os.path.join(PASTA_DRIVE, 'resultados_semanal_r2')
PASTA_UF_R2 = os.path.join(PASTA_DRIVE, 'resultados_uf_r2')
PASTA_CACHE = os.path.join(PASTA_DRIVE, 'cache_r2')
PASTA_AV2 = os.path.join(PASTA_DRIVE, 'avaliacao_r2')
PASTA_COMP = os.path.join(PASTA_DRIVE, 'resultados_competicao')
PASTA_EXT = os.path.join(PASTA_DRIVE, 'resultados_externos')
ARQ_SEL = os.path.join(PASTA_DRIVE, 'resultados_semanal', 'selecao_grupos.json')
for p in (PASTA_SEM_R2, PASTA_UF_R2, PASTA_CACHE, PASTA_AV2, os.path.join(PASTA_CACHE, 'ckpt')):
    os.makedirs(p, exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM, uf_lib as U, semanal_lib as S, r2_lib as R2
for m in (V, L, CM, U, S, R2):
    importlib.reload(m)
try:
    import torch
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma', '· CPUs:', os.cpu_count())
except ImportError:
    torch = None
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'),
                   os.path.join(PASTA_LOCAL, 'dados_uf'))
D = ctx.Du
TUNING, TESTE = U.TUNING, U.TESTE
RODADAS = S.rodadas(D, felipe=True)
LIM = R2.limiares(D)                        # p25/p75 de cada UF com dados até 2017 (antes da afinação)
REFAZER = False


In [ ]:
# @title 2 · Configurações
CK = os.path.join(PASTA_CACHE, 'ckpt')
base_t = dict(fabrica=lambda: U.MTGNN(), alvo='ancora', escala=False, peso_pop=0.5, cov=None)
TOT = (16, 67, 0.3)                       # D3: total das semanas 16–67 a partir da SE25 (≈ SE41→SE40), peso 0,3
CFG_T = [dict(base_t, nome='MTGNN_escala_pp_UF', escala=True),                                   # B1
         dict(base_t, nome='MTGNN_qalto_pp_UF', peso_q=R2.PESO_Q_ALTO),                          # D1
         dict(base_t, nome='MTGNN_total_pp_UF', tot=TOT),                                        # D3
         dict(base_t, nome='MTGNN_pre_pp_UF', pre=True, pre_nome='ancora', ep_pre=30, ep_ajuste=30),   # B2
         dict(base_t, nome='MTGNN_r2full_UF', escala=True, peso_q=R2.PESO_Q_ALTO, tot=TOT, pre=True,
              pre_nome='ancora_esc', ep_pre=30, ep_ajuste=30)]
base_s = dict(fabrica=lambda: U.MTGNN(), alvo='cresc', escala=False, cov=None, H=4)
CFG_S = [dict(base_s, nome='MTGNN_cresc_escala_sem', escala=True),
         dict(base_s, nome='MTGNN_cresc_qalto_sem', peso_q=R2.PESO_Q_ALTO),
         dict(base_s, nome='MTGNN_cresc_pre_sem', pre=True, pre_nome='cresc', ep_pre=30, ep_ajuste=30)]
MODELOS_T = {c['nome']: R2.RedeTemporadaR2(c, CK) for c in CFG_T}
MODELOS = [R2.RedeSemanalR2(c, CK) for c in CFG_S]
TEMPORADAS = TUNING + TESTE
list(MODELOS_T), [m.nome for m in MODELOS]


## Temporada (→ `resultados_uf_r2/`)


In [ ]:
# @title Rodar temporada (L.rodar: pula o que já existe; as redes retomam do último checkpoint)
erros = []
for nome, m in MODELOS_T.items():
    for s in TEMPORADAS:
        try:
            dt = L.rodar(ctx, m, s, PASTA_UF_R2, M=500, refazer=REFAZER)
            if dt is not None:
                print(f'✅ {nome:30s} {s}: {dt / 60:5.1f} min', flush=True)
        except Exception as e:
            erros.append((nome, s, repr(e)[:500]))
            print(f'❌ {nome:30s} {s}: {repr(e)[:300]}', flush=True)
            traceback.print_exc(limit=2)
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()
print('Erros:', len(erros))
pd.DataFrame(erros, columns=['modelo', 'temporada', 'erro']).to_csv(os.path.join(PASTA_UF_R2, f'erros_{NOTEBOOK}.csv'), index=False)


## Semanal (→ `resultados_semanal_r2/`)


In [ ]:
# @title Rodar (retoma de onde parou; um erro não interrompe o resto)
erros = []
for m in MODELOS:
    erros += S.rodar_semanal(ctx, m, RODADAS, PASTA_SEM_R2, refazer=REFAZER)
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
print('Erros:', len(erros))
pd.DataFrame(erros, columns=['modelo', 'rodada', 'erro']).to_csv(os.path.join(PASTA_SEM_R2, f'erros_{NOTEBOOK}.csv'), index=False)
feitos = {os.path.basename(f)[:-8] for f in glob.glob(os.path.join(PASTA_SEM_R2, '*.parquet'))}
pd.DataFrame({m.nome: {r: '●' if f'{m.nome}__{r}' in feitos else '·' for r in RODADAS} for m in MODELOS}).T
